# Valores Categóricos y Enums

In [ ]:
import polars as pl

## Datos Categóricos
- Algunas columnas de texto consisten en un número limitado de valores únicos.
- Estas columnas almacenan datos categóricos. Categórico significa "compuesto por categorías".
- Algunos ejemplos de datos categóricos: género, tipo de sangre, país, plan de suscripción.
- Por defecto, Polars almacena cada cadena de texto por separado en memoria, incluso si los valores son idénticos.

### El Tipo de Dato Categorical
- El tipo de dato `Categorical` utiliza una caché de cadenas internamente.
- La caché es un diccionario que mapea valores String a un entero complementario `UInt32`.
- El entero es la representación "física", mientras que el String es la representación "léxica" del valor.
- El diseño de caché almacena cada valor de cadena en memoria una sola vez. El entero se mapea de vuelta a la cadena.
- Los datos categóricos son ideales cuando hay un número pequeño de valores únicos en una columna.

In [ ]:
gym = pl.read_csv("gym_memberships.csv")
gym.head(5)

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,str,str,str,i64,bool
1,"""Silver""","""Pilates""","""Los Angeles""",20,false
2,"""Bronze""","""Spin""","""Miami""",7,true
3,"""Bronze""","""Pilates""","""Los Angeles""",9,false
4,"""Silver""","""Pilates""","""Dallas""",1,true
5,"""Gold""","""Pilates""","""Miami""",14,false


- El método `estimated_size` devuelve el número de bytes que el `DataFrame` ocupa en memoria.

In [ ]:
gym.estimated_size()

33678

- El `DataFrame` tiene 1000 filas y 6 columnas.
- Las columnas `membership_tier`, `favorite_class` y `city` almacenan un número pequeño de valores únicos.
- El método `n_unique` devuelve el número de valores únicos en una columna.

In [ ]:
gym.select(pl.all().n_unique())

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
u32,u32,u32,u32,u32,u32
1000,3,4,5,21,2


In [ ]:
gym.select(pl.col("membership_tier").unique())

membership_tier
str
"""Bronze"""
"""Gold"""
"""Silver"""


In [ ]:
gym.select(pl.col("favorite_class").unique())

favorite_class
str
"""Spin"""
"""HIIT"""
"""Pilates"""
"""Yoga"""


In [ ]:
gym.select(pl.col("city").unique())

city
str
"""Chicago"""
"""Dallas"""
"""Miami"""
"""Los Angeles"""
"""New York"""


- Vamos a convertir las columnas con un número pequeño de valores repetidos al tipo `Categorical`.

In [ ]:
gym = pl.read_csv(
    "gym_memberships.csv",
    schema_overrides={
        "membership_tier": pl.Categorical,
        "favorite_class": pl.Categorical,
        "city": pl.Categorical,
    },
)
gym.head(5)

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,cat,cat,cat,i64,bool
1,"""Silver""","""Pilates""","""Los Angeles""",20,false
2,"""Bronze""","""Spin""","""Miami""",7,true
3,"""Bronze""","""Pilates""","""Los Angeles""",9,false
4,"""Silver""","""Pilates""","""Dallas""",1,true
5,"""Gold""","""Pilates""","""Miami""",14,false


### ¿Por qué ahorramos memoria?

Cuando usamos `str`, cada celda almacena la cadena completa.
Cuando usamos `pl.Categorical`, Polars usa un mapeo:
- **Caché (Léxico):** `0: "Silver", 1: "Bronze", 2: "Gold"`
- **Columna (Físico):** `[0, 1, 1, 0, 2, ...]` (Enteros de 32 bits)

Vamos a inspeccionar la representación física (los enteros) de la columna `membership_tier`:

In [ ]:
# Mostramos la diferencia entre la vista lógica (texto) y la física (enteros)
print("Vista lógica (lo que vemos):")
display(gym.select("membership_tier").head(5))

print("\nVista física (cómo se guarda realmente en memoria):")
display(gym.select(pl.col("membership_tier").to_physical()).head(5))

Vista lógica (lo que vemos):


membership_tier
cat
"""Silver"""
"""Bronze"""
"""Bronze"""
"""Silver"""
"""Gold"""



Vista física (cómo se guarda realmente en memoria):


membership_tier
u32
9
3
3
9
0


- ¡El `DataFrame` ocupa un 17% menos de memoria!

In [ ]:
gym.estimated_size()

28128

In [ ]:
28128 / 33678

0.8352039907357919

### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/
- https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#creating-a-categorical-series
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.estimated_size.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.all.html
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Categorical.html

## Enums
- Los Enums son un tipo de dato similar a los categóricos. **Son ideales cuando los valores únicos se conocen de antemano**.
- La documentación de Polars recomienda usar enums cuando sea posible. El tipo `Categorical` tiene un costo de rendimiento en comparación.
- Se instancia un `Enum` con el constructor `pl.Enum`. Se le pasa una lista de los valores distintos.
- **Al convertir una columna a una columna enum, Polars lanzará un error si un valor no se encuentra dentro del enum**.
- El tipo `Categorical` no lanzará este error porque los valores no se conocen de antemano.

In [ ]:
pl.read_csv("gym_memberships.csv").estimated_size()

33678

In [ ]:
pl.read_csv(
    "gym_memberships.csv",
    schema_overrides={
        "membership_tier": pl.Categorical,
        "favorite_class": pl.Categorical,
        "city": pl.Categorical,
    },
).estimated_size()

28128

In [ ]:
tiers_enum = pl.Enum(["Bronze", "Silver", "Gold"])
cities_enum = pl.Enum(["Miami", "Chicago", "Los Angeles", "New York", "Dallas"])
classes_enum = pl.Enum(["HIIT", "Yoga", "Spin", "Pilates"])

In [ ]:
gym = pl.read_csv(
    "gym_memberships.csv",
    schema_overrides={
        "membership_tier": tiers_enum,
        "favorite_class": classes_enum,
        "city": cities_enum,
    },
)

gym.head(1)

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,enum,i64,bool
1,"""Silver""","""Pilates""","""Los Angeles""",20,false


La reducción es mayor con `pl.Enum` por cómo gestiona Polars la 'caché' de strings:

- **Categorical (Caché Global)**: Por defecto, los tipos `Categorical` comparten una infraestructura de caché que permite que diferentes columnas se comuniquen entre sí, pero esto añade metadatos y una estructura de datos más compleja para mantener esa flexibilidad.
- **Enum (Caché Local y Estática)**: Al usar `pl.Enum`, tú le dices a Polars exactamente qué valores existen de antemano. Esto permite a Polars crear una tabla de búsqueda mucho más compacta y 'privada' para esa columna, eliminando el exceso de equipaje (overhead) que requiere el sistema de categorías globales.

En resumen, el `Enum` es una versión optimizada y estricta del categórico: al saber qué datos esperar, Polars puede comprimir la información al máximo.

In [ ]:
gym.estimated_size()

19128

- ¡El tipo enum permite una reducción del 44% en memoria!

In [ ]:
19128 / 33678

0.567967218955995

- Los Enums permiten que cada categoría exista independientemente de las demás (caché separada).
- Internamente, Polars modela un enum como un categórico optimizado.
- Seguimos usando el espacio de nombres `cat` para acceder a los métodos categóricos/enum.

In [ ]:
gym.select(pl.col("membership_tier").cat.get_categories())

membership_tier
str
"""Bronze"""
"""Silver"""
"""Gold"""


In [ ]:
gym.select(pl.col("favorite_class").cat.get_categories())

favorite_class
str
"""HIIT"""
"""Yoga"""
"""Spin"""
"""Pilates"""


In [ ]:
gym.select(pl.col("city").cat.get_categories())

city
str
"""Miami"""
"""Chicago"""
"""Los Angeles"""
"""New York"""
"""Dallas"""


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#creating-an-enum
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.cat.get_categories.html

## Enums, Categóricos y Ordenamiento
- Hay algunos matices interesantes cuando optamos por almacenar valores categóricos/enum.

In [ ]:
tiers_enum = pl.Enum(["Bronze", "Silver", "Gold"])
cities_enum = pl.Enum(["Miami", "Chicago", "Los Angeles", "New York", "Dallas"])
classes_enum = pl.Enum(["HIIT", "Yoga", "Spin", "Pilates"])

gym = pl.read_csv(
    "gym_memberships.csv",
    schema_overrides={
        "membership_tier": tiers_enum,
        "favorite_class": classes_enum,
        "city": pl.Categorical,
    },
)
gym.head(1)

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,cat,i64,bool
1,"""Silver""","""Pilates""","""Los Angeles""",20,false


- El orden de las variantes en la declaración del enum determina el orden de clasificación de la columna categórica.
- Por ejemplo, declaramos nuestro orden de niveles como `"Bronze"`, luego `"Silver"`, luego `"Gold"`.
- Polars ordenará una columna según el orden de declaración de variantes, no en orden alfabético (léxico).
- Podemos usar esta lógica de ordenamiento a nuestro favor en dominios donde el orden no es alfabético.
- Por ejemplo, podemos _querer_ que `"Bronze"` aparezca antes que `"Silver"` en el orden de clasificación.

In [ ]:
gym.sort("membership_tier")

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,cat,i64,bool
2,"""Bronze""","""Spin""","""Miami""",7,true
3,"""Bronze""","""Pilates""","""Los Angeles""",9,false
6,"""Bronze""","""HIIT""","""New York""",8,false
7,"""Bronze""","""Pilates""","""New York""",11,false
8,"""Bronze""","""HIIT""","""Los Angeles""",2,false
…,…,…,…,…,…
983,"""Gold""","""HIIT""","""Los Angeles""",14,false
984,"""Gold""","""Spin""","""Miami""",8,false
987,"""Gold""","""Pilates""","""Miami""",9,false



- El orden de variantes también influye en operaciones relacionadas con el ordenamiento como `>` y `<`.
- El siguiente ejemplo filtra filas con un nivel de membresía mayor o igual a `"Silver"`.
- Las filas incluyen valores de nivel de membresía `"Silver"` y `"Gold"`.

In [ ]:
gym.filter(pl.col("membership_tier") >= "Silver")

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,cat,i64,bool
1,"""Silver""","""Pilates""","""Los Angeles""",20,false
4,"""Silver""","""Pilates""","""Dallas""",1,true
5,"""Gold""","""Pilates""","""Miami""",14,false
9,"""Silver""","""Yoga""","""Chicago""",14,false
11,"""Silver""","""Spin""","""Dallas""",7,true
…,…,…,…,…,…
990,"""Silver""","""Pilates""","""New York""",12,true
991,"""Silver""","""Yoga""","""Dallas""",18,false
992,"""Silver""","""Yoga""","""Miami""",3,true


- Con los categóricos, Polars usa ordenamiento léxico (alfabético).
- La razón es que Polars no conoce el orden definitivo de los valores.
- Los valores del categórico se construyen a medida que Polars los encuentra.

In [ ]:
gym.sort("city")

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,cat,i64,bool
9,"""Silver""","""Yoga""","""Chicago""",14,false
19,"""Bronze""","""Spin""","""Chicago""",19,false
32,"""Bronze""","""HIIT""","""Chicago""",9,true
39,"""Gold""","""Spin""","""Chicago""",13,true
40,"""Gold""","""Yoga""","""Chicago""",1,false
…,…,…,…,…,…
977,"""Gold""","""HIIT""","""New York""",9,true
988,"""Gold""","""Pilates""","""New York""",16,true
990,"""Silver""","""Pilates""","""New York""",12,true


- El filtrado también será alfabético.

In [ ]:
gym.filter(pl.col("city") > "Los Angeles")

member_id,membership_tier,favorite_class,city,visits_last_month,is_active
i64,enum,enum,cat,i64,bool
2,"""Bronze""","""Spin""","""Miami""",7,true
5,"""Gold""","""Pilates""","""Miami""",14,false
6,"""Bronze""","""HIIT""","""New York""",8,false
7,"""Bronze""","""Pilates""","""New York""",11,false
14,"""Bronze""","""Yoga""","""New York""",11,true
…,…,…,…,…,…
992,"""Silver""","""Yoga""","""Miami""",3,true
994,"""Bronze""","""Pilates""","""Miami""",16,false
995,"""Bronze""","""Spin""","""New York""",11,false


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#category-ordering-and-comparison
- https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#lexical-comparison-with-strings
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html